# cli

> `sysone train | eval | predict | serve | run | jobs | publish`: the library from the command line.

In [ ]:
#| default_exp cli

In [ ]:
#| export
import json, os, sys
from pathlib import Path
from typing import Optional

import typer

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
from typer.testing import CliRunner
import os, tempfile, logging
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

Each command is a few lines over the library, so anything the command line does, a script does the same way. The heavy imports happen inside the commands, so `sysone --help` is instant.

```{mermaid}
flowchart LR
    TR["sysone train"] --> T1["decision_learner (text or multimodal),<br/>lr_find, fit_one_cycle, calibrate, export"]
    EV["sysone eval"] --> T2["Decider.load, then evaluate"]
    PR["sysone predict"] --> T3["Decider.load, then predict"]
    SV["sysone serve"] --> T4["serve_app under uvicorn"]
    RU["sysone run"] --> T5["cloud.run on Kaggle or Colab"]
    JB["sysone jobs status"] --> T6["cloud.status"]
    PB["sysone publish"] --> T7["the Hub, or Kaggle Models"]
```

In [ ]:
#| export
app = typer.Typer(help="Jevify BERT-like encoders with sysone.", no_args_is_help=True, add_completion=False)
jobs_app = typer.Typer(help="Cloud jobs submitted with `sysone run`.", no_args_is_help=True)
app.add_typer(jobs_app, name="jobs")

def _data(data:str, valid, calib:float):
    "TypedDecisions from a JSON-lines file, a registry name or a Hub dataset"
    from sysone.data import TypedDecisions
    from sysone.datasets import REGISTRY
    v = float(valid) if valid is not None and valid.replace(".", "", 1).isdigit() else valid
    if Path(data).exists(): return TypedDecisions.from_json(data, valid=v, calib=calib)
    if data in REGISTRY: return TypedDecisions.from_registry(data, **({"valid": v} if v is not None else {}), calib=calib)
    return TypedDecisions.from_hub(data, valid=v, calib=calib)

## train

`sysone train DATA ENCODER` is the ten-line example: data from a JSON-lines file, a registry name or a Hub dataset, a learner for the encoder (the text application's defaults, or the multimodal one's for a multimodal encoder), one-cycle training, calibration, export.

In [ ]:
#| export
@app.command()
def train(data:str = typer.Argument(..., help="a .jsonl file, a registry name (typed_decisions) or a Hub dataset id"),
          encoder:str = typer.Argument("answerdotai/ModernBERT-large", help="encoder id or folder"),
          output:Optional[Path] = typer.Option(None, help="export folder (SYSONE_OUTPUT, else sysone-export)"),
          regime:str = typer.Option("head", "--train", help="head, lora, mica or full"),
          epochs:int = typer.Option(4, help="training epochs"),
          lr:Optional[float] = typer.Option(None, help="head learning rate (the encoder gets a quarter); lr_find's valley if omitted"),
          valid:Optional[str] = typer.Option(None, help="a split name or a fraction of the training cases"),
          calib:float = typer.Option(0.1, help="fraction of training cases held out for calibration"),
          cache:str = typer.Option("auto", help="auto, masks, rows or none"),
          loss:Optional[str] = typer.Option(None, help="soft_ce or rlcd (the data's default if omitted)"),
          preset:Optional[str] = typer.Option(None, help="hardware preset (SYSONE_PRESET or the machine's if omitted)")):
    "Train a decision model and export it"
    from sysone.models import EncoderSpec
    d = _data(data, valid, calib)
    multimodal = EncoderSpec.from_pretrained(encoder).modality == "multimodal"
    from sysone import multimodal as mm, text
    make = mm.decision_learner if multimodal else text.decision_learner
    learn = make(d, encoder, train=regime, cache=None if cache == "none" else cache, loss=loss, preset=preset)
    typer.echo(repr(learn))
    rate = lr if lr is not None else learn.lr_find().valley
    learn.fit_one_cycle(epochs, lr=rate)
    if d.cases.get("calib") is not None: learn.calibrate()
    if d.cases.get("valid") is not None: typer.echo(json.dumps({k: round(v, 4) for k, v in learn.validate().items()}))
    typer.echo(f"exported to {learn.export(output or Path(os.environ.get('SYSONE_OUTPUT', 'sysone-export')))}")

## eval and predict

In [ ]:
#| export
@app.command("eval")
def eval_(model:str = typer.Argument(..., help="an export folder, a Laya checkpoint or a Hub id"),
          data:str = typer.Option("typed_decisions", help="a .jsonl file with gold, a registry name, or a suite (ag_news, banking77, ...)"),
          split:str = typer.Option("test", help="split, for Hub data"),
          n:Optional[int] = typer.Option(None, help="evaluate the first n cases"),
          by:Optional[str] = typer.Option("workflow", help="report accuracy per value of this record field")):
    "Score a model on typed decisions or a zero-shot suite"
    from sysone.inference import Decider
    from sysone.evaluate import evaluate
    from sysone.datasets import SUITES, suite_records
    from sysone.data import read_jsonl
    jev = Decider.load(model)
    if Path(data).exists(): recs = read_jsonl(data)
    elif data in SUITES: recs, by = suite_records(data, n=n), None
    else:
        from datasets import load_dataset
        recs = list(load_dataset("LocalLLaMA/typed-decisions", "all", split=split)) if data == "typed_decisions" else list(load_dataset(data, split=split))
    recs = recs[:n] if n else recs
    res = evaluate(jev, recs, by=by if recs and by in recs[0] else None)
    typer.echo(json.dumps(res, indent=1, default=float))

@app.command()
def predict(model:str = typer.Argument(..., help="an export folder, a Laya checkpoint or a Hub id"),
            questions:Path = typer.Option(..., help="a JSON file of questions, {qid: {type, instructions, criteria}}"),
            state:Optional[str] = typer.Option(None, help="the state: text, JSON, or @file"),
            device:Optional[str] = typer.Option(None, help="cuda, mps or cpu")):
    "Answer typed questions about one state"
    from sysone.inference import Decider
    s = state if state is not None else sys.stdin.read()
    if s.startswith("@"): s = Path(s[1:]).read_text()
    try: s = json.loads(s)
    except json.JSONDecodeError: pass
    typer.echo(json.dumps(Decider.load(model, device=device).predict(s, json.loads(questions.read_text())), indent=1))

## serve

`sysone serve my-jev` runs the FastAPI app of `sysone.inference` (`POST /v1/systemone`, `GET /health`); it needs `sysone[serve]`. For Laya-format text exports `laya[serve]` serves the same route.

In [ ]:
#| export
@app.command()
def serve(model:str = typer.Argument(..., help="an export folder, a Laya checkpoint or a Hub id"),
          host:str = typer.Option("127.0.0.1"), port:int = typer.Option(8000), device:Optional[str] = typer.Option(None)):
    "Serve POST /v1/systemone and GET /health"
    import uvicorn
    from sysone.inference import Decider, serve_app
    uvicorn.run(serve_app(Decider.load(model, device=device)), host=host, port=port)

## run, jobs and publish

In [ ]:
#| export
@app.command()
def run(script:Path = typer.Argument(..., help="the training script, the same file that runs locally"),
        on:str = typer.Option("kaggle", help="kaggle or colab"),
        accelerator:Optional[str] = typer.Option(None, help="Kaggle machine: NvidiaTeslaT4 (2 GPUs), NvidiaL4, NvidiaTeslaA100, NvidiaH100"),
        gpu:Optional[str] = typer.Option(None, help="Colab GPU: T4, L4, A100, H100"),
        slug:Optional[str] = typer.Option(None, help="Kaggle kernel id, owner/name"),
        data:Optional[str] = typer.Option(None, help="passed to the script as SYSONE_DATA; kaggle:owner/dataset mounts a Kaggle dataset"),
        push:Optional[str] = typer.Option(None, help="Hub repo the export is pushed to at the end"),
        keep:bool = typer.Option(True, help="Colab: keep the VM to download the export"),
        dry_run:bool = typer.Option(False, "--dry-run", help="write the job files and print the command, submit nothing")):
    "Run a training script on Kaggle or Colab"
    from sysone import cloud
    job = cloud.run(script, on=on, accelerator=accelerator, gpu=gpu, slug=slug, data=data, push=push, keep=keep, dry_run=dry_run)
    typer.echo(json.dumps(job.__dict__, indent=1))

@jobs_app.command("status")
def jobs_status(refresh:bool = typer.Option(True, help="poll Kaggle for the current status")):
    "List submitted jobs"
    from sysone import cloud
    for j in cloud.status(refresh):
        typer.echo(f"{j.backend:7s} {j.ref:40s} {j.machine:16s} {j.status}")

@app.command()
def publish(model:Path = typer.Argument(..., help="an export folder"),
            to:str = typer.Option("hub", help="hub or kaggle"),
            repo:Optional[str] = typer.Option(None, help="Hub repo id (to hub)"),
            owner:Optional[str] = typer.Option(None, help="Kaggle owner (to kaggle)"),
            name:Optional[str] = typer.Option(None, help="Kaggle model slug (to kaggle)"),
            private:bool = typer.Option(True), dry_run:bool = typer.Option(False, "--dry-run")):
    "Publish an export to the Hub (the default) or to Kaggle Models"
    if to == "kaggle":
        from sysone import cloud
        typer.echo(json.dumps(cloud.publish_kaggle(model, owner, name or model.name, private=private, dry_run=dry_run)))
    elif to == "hub":
        if repo is None: raise typer.BadParameter("--repo is needed to publish to the Hub")
        if dry_run: typer.echo(f"would upload {model} to {repo}"); return
        from huggingface_hub import HfApi
        api = HfApi(); api.create_repo(repo, exist_ok=True, private=private)
        typer.echo(api.upload_folder(folder_path=str(model), repo_id=repo, commit_message="sysone export"))
    else: raise typer.BadParameter(f"unknown destination {to!r}")

def main(): app()

## Trying it

On the fixtures, end to end: train, predict, evaluate, and a dry run of a cloud job:

In [ ]:
runner = CliRunner()
r = runner.invoke(app, ["--help"])
test_eq(r.exit_code, 0)
print(r.output)

                                                                                
 Usage: root [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
 Jevify BERT-like encoders with sysone.                                         
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help          Show this message and exit.                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ train    Train a decision model and export it                                │
│ eval     Score a model on typed decisions or a zero-shot suite               │
│ predict  Answer typed questions about one state                              │
│ serve    Serve POST /v1/sy

In [ ]:
out = Path(tempfile.mkdtemp()) / "cli-jev"
os.environ["SYSONE_CACHE"] = tempfile.mkdtemp()
r = runner.invoke(app, ["train", "fixtures/typed_decisions_tiny.jsonl", "fixtures/tiny-modernbert", "--output", str(out),
                        "--epochs", "2", "--lr", "3e-3", "--valid", "0.25", "--preset", "cpu"])
if r.exit_code != 0: print(r.output); raise r.exception
test_eq((out / "sysone.json").exists(), True)
print(r.output[-400:])

_per_second': '1977', 'train_steps_per_second': '273.7', 'train_loss': '0.9173', 'epoch': '2'}
{"accuracy": 0.36, "soft_accuracy": 0.3767, "brier": 0.26, "kl": 0.4434, "tv": 0.366, "ece": 0.1159, "score_mae": 0.8379, "within_one": 0.6, "n": 25, "accuracy_choice": 0.4286, "accuracy_score": 0.3, "accuracy_noul": 0.375}
exported to /var/folders/ym/y7_gxdb90s312g8bz06h3qqm0000gn/T/tmpcjz36ll1/cli-jev



In [ ]:
qf = Path(tempfile.mkdtemp()) / "q.json"
qf.write_text(json.dumps({"urgent": {"type": "noul", "instructions": "This needs attention today."}}))
r = runner.invoke(app, ["predict", str(out), "--questions", str(qf), "--state", "The server room is flooding.", "--device", "cpu"])
test_eq(r.exit_code, 0)
test_eq(set(json.loads(r.output)["urgent"]), {"type", "noul", "confidence", "answer_confidence"})

In [ ]:
r = runner.invoke(app, ["eval", str(out), "--data", "fixtures/typed_decisions_tiny.jsonl", "--n", "4"])
test_eq((r.exit_code, json.loads(r.output)["n"]), (0, 20))

In [ ]:
os.environ["SYSONE_JOBS"] = str(Path(tempfile.mkdtemp()) / "jobs.json")
os.environ["SYSONE_JOBS_DIR"] = tempfile.mkdtemp()
script = Path(tempfile.mkdtemp()) / "train.py"; script.write_text("print('hi')\n")
r = runner.invoke(app, ["run", str(script), "--on", "kaggle", "--slug", "me/sysone-cli", "--push", "me/jev", "--dry-run"])
test_eq((r.exit_code, json.loads(r.output)["machine"]), (0, "NvidiaTeslaT4"))
test_eq(runner.invoke(app, ["publish", str(out), "--dry-run", "--repo", "me/jev"]).output.strip(), f"would upload {out} to me/jev")
r = runner.invoke(app, ["publish", str(out), "--to", "kaggle", "--owner", "me", "--name", "my-jev", "--dry-run"])
test_eq((r.exit_code, json.loads(r.output)["variation"][:4]), (0, ["kaggle", "models", "variations", "create"]))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()